## Features extraction from description 


In [1]:
import pandas as pd
import numpy as np
import json

df = pd.read_csv(r"../Data/raw/airbnb_processed.csv", encoding="utf-8")
df.shape

(831, 31)

In [2]:
df[['id','description']].head()

,id,description
0,1292713234154945394,Enjoy your stay with Panoramic View of the Giz...
1,1508718511630646313,Welcome To Marhaba Pyramids View Hotel✨Wake up...
2,1297327219631789358,The place is spacious and can accommodate more...
3,1606001853199411128,A designer retreat where ancient soul meets mo...
4,1314833467096489875,Enjoy stay in Single RoomTHE ROOM FEATURES1 Qu...


In [3]:
df['description'] = df['description'].astype(str)

Calculate the description length

In [4]:
# Add a column with the description length
df['description_length'] = df['description'].apply(lambda x: len(str(x).split()))
df[['description', 'description_length']].head()

,description,description_length
0,Enjoy your stay with Panoramic View of the Giz...,328
1,Welcome To Marhaba Pyramids View Hotel✨Wake up...,80
2,The place is spacious and can accommodate more...,98
3,A designer retreat where ancient soul meets mo...,249
4,Enjoy stay in Single RoomTHE ROOM FEATURES1 Qu...,119


### Features Extraction

#### Using `spacy` 
for extraction 'has_wifi' and 'has_pool' boolean features


In [5]:
# spaCy/keyword-based extraction: WiFi and Pool features

def extract_bool_feature(text):
    if pd.isnull(text):
        return pd.Series({'has_wifi': None, 'has_pool': None})
    text_lower = str(text).lower()
    
    # WiFi detection
    wifi_keywords = ['wifi', 'wi-fi', 'wireless internet', 'internet access', 'free wifi', 'fast wifi', 'high-speed wifi']
    has_wifi = any(kw in text_lower for kw in wifi_keywords)
    
    # Pool detection
    pool_keywords = ['pool', 'swimming pool', 'private pool', 'outdoor pool', 'indoor pool', 'jacuzzi', 'hot tub']
    has_pool = any(kw in text_lower for kw in pool_keywords)
    
    return pd.Series({'has_wifi': has_wifi, 'has_pool': has_pool})

# Apply to DataFrame
df[['has_wifi', 'has_pool']] = df['description'].apply(extract_bool_feature)
df[['description', 'has_wifi', 'has_pool']].head()

,description,has_wifi,has_pool
0,Enjoy your stay with Panoramic View of the Giz...,True,True
1,Welcome To Marhaba Pyramids View Hotel✨Wake up...,False,True
2,The place is spacious and can accommodate more...,False,False
3,A designer retreat where ancient soul meets mo...,True,True
4,Enjoy stay in Single RoomTHE ROOM FEATURES1 Qu...,False,False


#### Using `llama3.2:1b` 
for extracting 'view_type' , 'listing_tier', proximity_to_landmarks', 'target_guest' features from the rental **description** and **title**

In [6]:
import pandas as pd
import ollama
import json
import re

OLLAMA_MODEL = 'llama3.2:1b'
EXPECTED_KEYS = ['view_type', 'listing_tier', 'proximity_to_landmarks', 'target_guest']

def classify_listing(title, description):
    prompt = f'''Analyze this Airbnb listing in Egypt and return ONLY a JSON object.

Title: {title}
Description: {str(description)[:500]}

Return exactly this format:
{{
  "view_type": "pyramid_view | nile_view | sea_view | city_view | garden_view | pool_view | no_view",
  "listing_tier": "budget | mid_range | luxury",
  "proximity_to_landmarks": "walking_distance | nearby | far | not_mentioned",
  "target_guest": "family | couple | solo | business | any"
}}

JSON only, no explanation, no markdown.'''.strip()

    default_res = {k: "not_mentioned" if k == "proximity_to_landmarks" else "any" for k in EXPECTED_KEYS}

    try:
        response = ollama.generate(
            model=OLLAMA_MODEL,
            prompt=prompt,
            format='json'
        )
        
        # Parse response
        data = json.loads(response['response'])
        # Ensure all keys exist in the returned dictionary
        return {k: data.get(k, default_res[k]) for k in EXPECTED_KEYS}
        
    except Exception as e:
        print(f"Error processing row: {e}")
        return default_res

results = []

print(f"Starting extraction for {len(df)} rows...")

for i, (idx, row) in enumerate(df.iterrows()):
    result = classify_listing(row['title'], row['description'])
    results.append(result)

    # Output status every 10 extractions
    if (i + 1) % 10 == 0:
        print(f"Processed {i + 1}/{len(df)} listings...")

# Convert results list of dicts to a DataFrame
results_df = pd.DataFrame(results)

# Concatenate with the original dataframe
# Using .reset_index(drop=True) ensures indices align if df was filtered
df_final = pd.concat([df.reset_index(drop=True), results_df], axis=1)

# Save final results
df_final.to_csv('airbnb_with_features.csv', index=False)

print("\n--- Done! Summary of Extracted Features ---")
for col in EXPECTED_KEYS:
    print(f"\nValue counts for {col}:")
    print(df_final[col].value_counts())

Starting extraction for 831 rows...
Processed 10/831 listings...
Processed 20/831 listings...
Processed 30/831 listings...
Processed 40/831 listings...
Processed 50/831 listings...
Processed 60/831 listings...
Processed 70/831 listings...
Processed 80/831 listings...
Processed 90/831 listings...
Processed 100/831 listings...
Processed 110/831 listings...
Processed 120/831 listings...
Processed 130/831 listings...
Processed 140/831 listings...
Processed 150/831 listings...
Processed 160/831 listings...
Processed 170/831 listings...
Processed 180/831 listings...
Processed 190/831 listings...
Processed 200/831 listings...
Processed 210/831 listings...
Processed 220/831 listings...
Processed 230/831 listings...
Processed 240/831 listings...
Processed 250/831 listings...
Processed 260/831 listings...
Processed 270/831 listings...
Processed 280/831 listings...
Processed 290/831 listings...
Processed 300/831 listings...
Processed 310/831 listings...
Processed 320/831 listings...
Processed 330

In [8]:
df_final.head()

,id,title,url,thumbnail,lat,lng,rating_overall,reviews_count,rating_accuracy,rating_cleanliness,...,bedrooms,bathrooms,description_length,has_wifi,has_pool,has_pyramid_view,view_type,listing_tier,proximity_to_landmarks,target_guest
0,1292713234154945394,"ETERNA.Suite W Jaccuzi, Pyramids View & Balcony",https://www.airbnb.com/rooms/12927132341549453...,https://a0.muscache.com/im/pictures/hosting/Ho...,29.973873,31.146603,4.95,133.0,4.96,4.92,...,1,1.0,328,True,True,False,pyramid_view,budget,walking_distance,any
1,1508718511630646313,king khufu suite,https://www.airbnb.com/rooms/15087185116306463...,https://a0.muscache.com/im/pictures/hosting/Ho...,29.986300,31.143100,5.00,62.0,5.00,5.00,...,1,1.0,80,False,True,True,pyramid_view,mid_range,walking_distance,couple
2,1297327219631789358,Akasia Pyramids View,https://www.airbnb.com/rooms/12973272196317893...,https://a0.muscache.com/im/pictures/hosting/Ho...,29.978100,31.145400,4.91,176.0,4.91,4.91,...,1,1.0,98,False,False,True,pyramid_view,budget,nearby,family
3,1606001853199411128,Mountain Cave | Pyramids View & Jacuzzi,https://www.airbnb.com/rooms/16060018531994111...,https://a0.muscache.com/im/pictures/hosting/Ho...,29.979090,31.146920,4.92,12.0,5.00,4.75,...,1,1.0,249,True,True,True,pyramid_view,mid_range,nearby,family
4,1314833467096489875,Heaven of Pyramids,https://www.airbnb.com/rooms/13148334670964898...,https://a0.muscache.com/im/pictures/miso/Hosti...,29.978787,31.144191,4.71,125.0,4.74,4.61,...,1,1.0,119,False,False,True,pyramid_view,budget,walking_distance,family


In [10]:
df_final.to_csv('../Data/raw/airbnb_processed.csv',index=False, encoding="utf-8-sig")

In [12]:
df_final.drop(columns=["description", "thumbnail","url", "id", "price_price"], inplace=True)

In [13]:
df_final.to_csv('../Data/clean/airbnb_cleaned.csv',index=False, encoding="utf-8-sig")

#### Extracted Features Cleaning

In [2]:
import pandas as pd
df_final = pd.read_csv('../Data/clean/airbnb_cleaned.csv', encoding="utf-8")
df_final.head()

,title,lat,lng,rating_overall,reviews_count,rating_accuracy,rating_cleanliness,rating_value,rating_location,price_breakdown_baseprice_price,...,bedrooms,bathrooms,description_length,has_wifi,has_pool,has_pyramid_view,view_type,listing_tier,proximity_to_landmarks,target_guest
0,"ETERNA.Suite W Jaccuzi, Pyramids View & Balcony",29.973873,31.146603,4.95,133.0,4.96,4.92,4.92,4.79,750.00,...,1,1.0,328,True,True,False,pyramid_view,budget,walking_distance,any
1,king khufu suite,29.986300,31.143100,5.00,62.0,5.00,5.00,4.97,4.85,526.87,...,1,1.0,80,False,True,True,pyramid_view,mid_range,walking_distance,couple
2,Akasia Pyramids View,29.978100,31.145400,4.91,176.0,4.91,4.91,4.95,4.78,193.18,...,1,1.0,98,False,False,True,pyramid_view,budget,nearby,family
3,Mountain Cave | Pyramids View & Jacuzzi,29.979090,31.146920,4.92,12.0,5.00,4.75,4.58,4.75,592.44,...,1,1.0,249,True,True,True,pyramid_view,mid_range,nearby,family
4,Heaven of Pyramids,29.978787,31.144191,4.71,125.0,4.74,4.61,4.80,4.66,116.10,...,1,1.0,119,False,False,True,pyramid_view,budget,walking_distance,family


In [5]:
cols = ['view_type', 'listing_tier', 'proximity_to_landmarks', 'target_guest']

for col in cols:
    print(f"{col}:")
    print(df_final[col].unique())
    print("-" * 40)

view_type:
['pyramid_view' 'pyramid_view | nile_view | sea_view'
 'pyramid_view | nile_view | sea_view | city_view | garden_view | pool_view | no_view'
 "['pyramid_view', 'nile_view', 'sea_view']"
 'pyramid_view | nile_view | sea_view | city_view | garden_view | pool_view'
 "['pyramid_view', 'garden_view']" "['pyramid_view', 'nile_view']"
 "['pyramid_view', 'nile_view', 'sea_view', 'city_view', 'garden_view']"
 'pyramid_view | nile_view'
 'pyramid_view | nile_view | sea_view | city_view | garden_view'
 'pyramid_view | nile_view | sea_view | city_view' 'pool_view | no_view'
 'garden_view' 'pool_view' 'city_view' 'lake_view' 'poolside'
 "['pyramid_view', 'nile_view', 'sea_view', 'city_view', 'garden_view', 'pool_view']"
 "['nile_view', 'pyramid_view']" 'sea_view' 'nile_view'
 "['pyramid_view', 'giza_pyramids_view']" "['nile_view', 'river_view']"
 'golf_view' "['garden_view', 'pool_view']"
 "['pyramid_view', 'nile_view', 'city_view']"
 "['pyramid_view', 'nile_view', 'sea_view', 'city_view

In [6]:
import ast
import re

def parse_to_list(value):
    if pd.isna(value):
        return []
    
    value = str(value).strip()
    
    # Case 1: already looks like a list
    if value.startswith("[") and value.endswith("]"):
        try:
            return [v.strip().lower() for v in ast.literal_eval(value)]
        except:
            pass
    
    # Case 2: weird dict → ignore
    if value.startswith("{") and value.endswith("}"):
        return []
    
    # Case 3: pipe-separated
    if "|" in value:
        return [v.strip().lower() for v in value.split("|")]
    
    # Case 4: single value
    return [value.lower()]

cols = ['view_type', 'listing_tier', 'proximity_to_landmarks', 'target_guest']

for col in cols:
    df_final[col] = df_final[col].apply(parse_to_list)

In [8]:
def clean_view(values):
    mapping = {
        'river_view': 'nile_view',
        'poolside': 'pool_view',
        'giza_pyramids_view': 'pyramid_view'
    }
    return list(set(mapping.get(v, v) for v in values))


def clean_guest(values):
    mapping = {
        'sojo': 'solo',
        'solitary business': 'business',
        'any': 'any'
    }
    return list(set(mapping.get(v, v) for v in values))


def clean_proximity(values):
    mapping = {
        'nearly': 'nearby',
        'nearly_a_part_of_a_city': 'nearby'
    }
    return list(set(mapping.get(v, v) for v in values))

In [9]:
df_final['view_type'] = df_final['view_type'].apply(clean_view)
df_final['target_guest'] = df_final['target_guest'].apply(clean_guest)
df_final['proximity_to_landmarks'] = df_final['proximity_to_landmarks'].apply(clean_proximity)

In [13]:
cols = ['view_type', 'listing_tier', 'proximity_to_landmarks', 'target_guest']

for col in cols:
    unique_vals = sorted(set(v for row in df_final[col] for v in row))
    print(f"{col}:")
    print(unique_vals)
    print("-" * 40)

view_type:
['city_view', 'garden_view', 'golf_view', 'lake_view', 'nile_view', 'no_view', 'pool_view', 'pyramid_view', 'sea_view']
----------------------------------------
listing_tier:
['budget', 'luxury', 'mid_range']
----------------------------------------
proximity_to_landmarks:
['city_view', 'far', 'nearby', 'nearer', 'no_view', 'not_mentioned', 'walking_distance']
----------------------------------------
target_guest:
['any', 'business', 'couple', 'family', 'solo', 'solosingle']
----------------------------------------


In [16]:
from collections import Counter

for col in cols:
    counter = Counter(v for row in df_final[col] for v in row)
    print(f"\n{col}:")
    for k, v in counter.most_common():
        print(f"{k}: {v}")


view_type:
pyramid_view: 787
nile_view: 318
sea_view: 230
city_view: 213
garden_view: 194
pool_view: 189
no_view: 156
lake_view: 2
golf_view: 1

listing_tier:
mid_range: 459
budget: 405
luxury: 159

proximity_to_landmarks:
walking_distance: 421
far: 243
nearby: 200
not_mentioned: 180
no_view: 7
city_view: 2
nearer: 1

target_guest:
family: 574
any: 241
couple: 233
business: 132
solo: 113
solosingle: 1


In [3]:
import pandas as pd
df_final = pd.read_csv('../Data/clean/airbnb_cleaned.csv', encoding="utf-8")
df_final.head()

,title,lat,lng,rating_overall,reviews_count,rating_accuracy,rating_cleanliness,rating_value,rating_location,price_breakdown_baseprice_price,...,bedrooms,bathrooms,description_length,has_wifi,has_pool,has_pyramid_view,view_type,listing_tier,proximity_to_landmarks,target_guest
0,"ETERNA.Suite W Jaccuzi, Pyramids View & Balcony",29.973873,31.146603,4.95,133.0,4.96,4.92,4.92,4.79,750.00,...,1,1.0,328,True,True,False,['pyramid_view'],['budget'],['walking_distance'],['any']
1,king khufu suite,29.986300,31.143100,5.00,62.0,5.00,5.00,4.97,4.85,526.87,...,1,1.0,80,False,True,True,['pyramid_view'],['mid_range'],['walking_distance'],['couple']
2,Akasia Pyramids View,29.978100,31.145400,4.91,176.0,4.91,4.91,4.95,4.78,193.18,...,1,1.0,98,False,False,True,['pyramid_view'],['budget'],['nearby'],['family']
3,Mountain Cave | Pyramids View & Jacuzzi,29.979090,31.146920,4.92,12.0,5.00,4.75,4.58,4.75,592.44,...,1,1.0,249,True,True,True,['pyramid_view'],['mid_range'],['nearby'],['family']
4,Heaven of Pyramids,29.978787,31.144191,4.71,125.0,4.74,4.61,4.80,4.66,116.10,...,1,1.0,119,False,False,True,['pyramid_view'],['budget'],['walking_distance'],['family']


In [5]:
df_final.drop(columns=["has_pyramid_view"], inplace=True)

In [6]:
df_final.to_csv('../Data/clean/airbnb_cleaned.csv',index=False, encoding="utf-8-sig")